# PDE time series to generators with confidence

**Current surface:** V0.19.

## Purpose

Quickstart for the V0.19 surface: start from a canonical PDE time series, compute derivatives and residuals, fit a translation generator, verify it on held-out data, and read the confidence evidence.

## What you will learn

- How `FieldBatch`, `DerivativeBatch`, `ResidualBatch`, `GeneratorFamily`, and `VerificationReport` fit together.
- How metadata tags keep residual evaluators honest.
- How direct SVD evidence, fallback status, span distance, and verification errors should be read together.
- How the same pipeline covers Heat, Fisher-KPP reaction-diffusion, and the V0.19 advection-diffusion strong path.

## Required extras

Core install is enough for the code; Matplotlib is used only for optional tutorial plots and is included in `.[test]`.

## Expected runtime

About 1-2 minutes on a laptop.

## Out of scope

No new PDEs, no KS public API, no weak-form workflow, no train/test policy, and no claim that empirical verification is a mathematical proof.

These notebooks are tutorials, not API contracts. Example outputs are runtime summaries, not canonical paper artifacts.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np

from notebooks._tutorial_utils import (
    confidence_card,
    field_snapshot,
    plot_field_heatmap,
    plot_singular_values,
    plot_verification_curve,
    pretty_json,
    print_cards,
)
from pdelie.data import (
    generate_heat_1d_field_batch,
    generate_reaction_diffusion_1d_field_batch,
    split_batch_train_heldout,
)
from pdelie.derivatives import compute_spectral_fd_derivatives
from pdelie.reporting import (
    summarize_generator_fit_diagnostics,
    summarize_residual_batch,
    summarize_verification_report,
    summarize_vertical_slice,
)
from pdelie.residuals import HeatResidualEvaluator, ReactionDiffusionResidualEvaluator
from pdelie.symmetry import fit_translation_generator
from pdelie.verification import verify_translation_generator

CONFIG = {
    "fit_epsilon": 1e-4,
    "verification_epsilons": None,
    "span_tolerance": 5e-2,
    "heat_seed_train": 100,
    "heat_seed_heldout": 101,
    "reaction_diffusion_seed": 18018,
    "reaction_diffusion_split_seed": 18019,
}
CONFIG

## 1. Canonical fields are the common currency

`FieldBatch` is the object that lets data generation, residual evaluation, fitting, verification, and downstream helpers agree on axes, coordinates, variables, metadata, and provenance.


In [ ]:
heat_train = generate_heat_1d_field_batch(batch_size=4, num_times=33, num_points=64, seed=CONFIG["heat_seed_train"])
heat_heldout = generate_heat_1d_field_batch(batch_size=3, num_times=33, num_points=64, seed=CONFIG["heat_seed_heldout"])

print(pretty_json(field_snapshot(heat_train), max_chars=2000))
plot_field_heatmap(heat_train, title="Heat training sample")

## 2. Residuals define the scientific target

PDELie does not fit symmetries in a vacuum. A generator is meaningful only relative to the residual it preserves.


In [ ]:
heat_derivatives = compute_spectral_fd_derivatives(heat_train)
heat_evaluator = HeatResidualEvaluator()
heat_residual = heat_evaluator.evaluate(heat_train, heat_derivatives)
heat_residual_summary = summarize_residual_batch(heat_residual)
heat_residual_summary


## 3. Fit, verify, then inspect the confidence card

The fitted object is a `GeneratorFamily`. The confidence card summarizes residual health, fit conditioning, span evidence, and held-out finite-transform verification.


In [ ]:
heat_generator = fit_translation_generator(
    heat_train,
    heat_evaluator,
    epsilon=CONFIG["fit_epsilon"],
)
heat_verification = verify_translation_generator(
    heat_heldout,
    heat_generator,
    heat_evaluator,
    epsilon_values=CONFIG["verification_epsilons"],
    span_tolerance=CONFIG["span_tolerance"],
)

heat_fit_summary = summarize_generator_fit_diagnostics(heat_generator)
heat_verification_summary = summarize_verification_report(heat_verification)
heat_vertical_slice = summarize_vertical_slice(
    derivatives=heat_derivatives,
    residual=heat_residual,
    generator=heat_generator,
    verification=heat_verification,
    extra_metrics={"case": "heat_quickstart"},
)
heat_card = confidence_card(
    label="heat fitted translation",
    residual=heat_residual_summary,
    fit=heat_fit_summary,
    verification=heat_verification_summary,
)
print_cards([heat_card])
print(pretty_json({
    "vertical_slice_summary_type": heat_vertical_slice["summary_type"],
    "nested_sections": ["residual", "generator", "verification"],
}, max_chars=1500))
plot_singular_values(heat_fit_summary, title="Heat fit singular values")
plot_verification_curve(heat_verification_summary, title="Heat held-out verification")

## 4. Same workflow, Fisher-KPP reaction-diffusion

V0.18 added the scalar 1D periodic Fisher-KPP reaction-diffusion path, and V0.19 adds a similarly scoped constant-coefficient advection-diffusion path. Both use the existing order-2 derivative backend and ship only with direct SVD translation evidence, not reference-fallback evidence.

In [ ]:
rd_field = generate_reaction_diffusion_1d_field_batch(batch_size=5, seed=CONFIG["reaction_diffusion_seed"])
rd_train, rd_heldout = split_batch_train_heldout(
    rd_field,
    train_size=2,
    seed=CONFIG["reaction_diffusion_split_seed"],
)
rd_evaluator = ReactionDiffusionResidualEvaluator()
rd_derivatives = compute_spectral_fd_derivatives(rd_train)
rd_residual = rd_evaluator.evaluate(rd_train, rd_derivatives)
rd_generator = fit_translation_generator(rd_train, rd_evaluator, epsilon=CONFIG["fit_epsilon"])
rd_verification = verify_translation_generator(rd_heldout, rd_generator, rd_evaluator)

rd_residual_summary = summarize_residual_batch(rd_residual)
rd_fit_summary = summarize_generator_fit_diagnostics(rd_generator)
rd_verification_summary = summarize_verification_report(rd_verification)
rd_card = confidence_card(
    label="Fisher-KPP fitted translation",
    residual=rd_residual_summary,
    fit=rd_fit_summary,
    verification=rd_verification_summary,
)
print_cards([heat_card, rd_card])
print(pretty_json({
    "equation_tag": rd_train.metadata["parameter_tags"]["equation"],
    "direct_svd_expected": rd_fit_summary["evidence_label"] == "direct_svd_in_tolerance",
    "reference_fallback_used": rd_fit_summary["reference_fallback_used"],
}, max_chars=2000))
plot_verification_curve(rd_verification_summary, title="Fisher-KPP held-out verification")

## Recap

You saw the core V0.19 flow: canonical field, derivatives, residual, fitted translation generator, held-out verification, and a compact confidence card.

## Common pitfalls

- Treating generator coefficients as enough evidence by themselves.
- Ignoring metadata tags; residual evaluators use them to reject the wrong equation family.
- Confusing empirical verification with a mathematical proof.
- Treating fallback-backed evidence as equivalent to direct SVD recovery.

## Extension ideas

- Swap Heat for Burgers, KdV, or Fisher-KPP and compare residual and fit diagnostics.
- Change `fit_epsilon` and inspect the condition number and span distance.
- Route the vertical-slice summary into a report table for your own experiment log.

## What to read/run next

Run `01_raw_vs_translation_canonical_discovery.ipynb` for downstream preprocessing, then `06_orbit_coverage_diagnostics.ipynb` for invariant/orbit reports.